# World Cup Predictor — Exploratory Data Analysis

This notebook explores international match results used to train the WC 2026 predictor.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.data_loader import clean_data, download_data, load_results
from src.elo import EloRating

sns.set_theme(style="whitegrid")
download_data()
df = clean_data(load_results())

## 1. Basic stats

Check dataset shape, date range, null counts, and tournament coverage.

In [ ]:
print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")
print("Null counts:")
print(df.isna().sum()[lambda s: s > 0])
print(f"Unique tournaments: {df['tournament'].nunique()}")

## 2. Matches per year (1990–2026)

Look for growth in international football activity and data gaps.

In [ ]:
year_counts = df.groupby(df["date"].dt.year).size()
year_counts = year_counts[(year_counts.index >= 1990) & (year_counts.index <= 2026)]
plt.figure(figsize=(12, 4))
year_counts.plot(kind="bar", color="steelblue")
plt.title("International matches per year")
plt.xlabel("Year")
plt.ylabel("Matches")
plt.tight_layout()
plt.show()

## 3. Top 20 teams by matches played

Identifies the most active national teams in the dataset.

In [ ]:
home_counts = df["home_team"].value_counts()
away_counts = df["away_team"].value_counts()
total_matches = (home_counts.add(away_counts, fill_value=0)).sort_values(ascending=False).head(20)
plt.figure(figsize=(10, 6))
total_matches.sort_values().plot(kind="barh", color="darkgreen")
plt.title("Top 20 teams by total matches")
plt.xlabel("Matches")
plt.tight_layout()
plt.show()

## 4. Home advantage overall

Compare home win, draw, and away win rates across all matches.

In [ ]:
outcome_pct = df["result"].value_counts(normalize=True).reindex(["H", "D", "A"]) * 100
plt.figure(figsize=(6, 4))
outcome_pct.plot(kind="bar", color=["#2ecc71", "#95a5a6", "#e74c3c"])
plt.title("Overall home advantage")
plt.ylabel("Percentage")
plt.xticks([0, 1, 2], ["Home win", "Draw", "Away win"], rotation=0)
plt.tight_layout()
plt.show()

## 5. Home advantage by tournament type

See whether home advantage differs in World Cups, friendlies, and other competitions.

In [ ]:
def tournament_type(t):
    if t == "FIFA World Cup":
        return "World Cup"
    if "Friendly" in str(t):
        return "Friendly"
    return "Other"

typed = df.copy()
typed["tournament_type"] = typed["tournament"].map(tournament_type)
pivot = typed.groupby(["tournament_type", "result"]).size().unstack(fill_value=0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0)[ ["H", "D", "A"] ]
pivot_pct.plot(kind="bar", figsize=(8, 4), color=["#2ecc71", "#95a5a6", "#e74c3c"])
plt.title("Home advantage by tournament type")
plt.ylabel("Share")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Goals per game by decade

Track scoring trends over time.

In [ ]:
goals = df.copy()
goals["total_goals"] = goals["home_score"] + goals["away_score"]
goals["decade"] = (goals["date"].dt.year // 10) * 10
decade_avg = goals.groupby("decade")["total_goals"].mean()
plt.figure(figsize=(8, 4))
decade_avg.plot(marker="o")
plt.title("Average goals per game by decade")
plt.xlabel("Decade")
plt.ylabel("Goals per game")
plt.tight_layout()
plt.show()

## 7. Goal difference distribution

Most matches are close; large blowouts are rare.

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(df["goal_diff"], bins=30, kde=True)
plt.title("Distribution of goal difference (home perspective)")
plt.xlabel("Goal difference")
plt.tight_layout()
plt.show()

## 8. Most common World Cup scorelines

Typical low-scoring patterns in World Cup history.

In [ ]:
wc = df[df["is_world_cup"]].copy()
wc["scoreline"] = wc["home_score"].astype(str) + "-" + wc["away_score"].astype(str)
wc["scoreline"].value_counts().head(15)

## 9. Top 10 teams' Elo over time (2000–2026)

Relative strength of major nations over the modern era.

In [ ]:
elo = EloRating()
elo.compute_all_ratings(df)
history = elo.get_ratings_history_df()
long_rows = []
for _, row in history.iterrows():
    long_rows.append({"date": row["date"], "team": row["home_team"], "elo": row["home_elo_after"]})
    long_rows.append({"date": row["date"], "team": row["away_team"], "elo": row["away_elo_after"]})
elo_long = pd.DataFrame(long_rows)
elo_long = elo_long[elo_long["date"] >= "2000-01-01"]
top_teams = elo.get_latest_ratings()
top10 = sorted(top_teams, key=top_teams.get, reverse=True)[:10]
subset = elo_long[elo_long["team"].isin(top10)]
plt.figure(figsize=(12, 6))
for team in top10:
    team_data = subset[subset["team"] == team].sort_values("date")
    plt.plot(team_data["date"], team_data["elo"], label=team)
plt.title("Top 10 teams by Elo (2000–2026)")
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

## 10. Feature correlation heatmap

Identify redundant or strongly related model inputs.

In [ ]:
features_path = ROOT / "data" / "processed" / "features.csv"
if features_path.exists():
    features = pd.read_csv(features_path)
    numeric = features.select_dtypes(include=[np.number]).drop(columns=["outcome", "home_goals", "away_goals"], errors="ignore")
    corr = numeric.corr()
    plt.figure(figsize=(14, 10))
    sns.heatmap(corr, cmap="coolwarm", center=0, linewidths=0.1)
    plt.title("Feature correlation heatmap")
    plt.tight_layout()
    plt.show()
else:
    print("Run src/features.py first to generate features.csv")

## 11. Recent form vs outcome

Teams with stronger recent form (points per game) should win more often.

In [ ]:
if features_path.exists():
    plot_df = features.dropna(subset=["recent_points_H", "outcome"]).copy()
    plot_df["outcome_label"] = plot_df["outcome"].map({0: "Home win", 1: "Draw", 2: "Away win"})
    plt.figure(figsize=(8, 4))
    sns.boxplot(data=plot_df, x="outcome_label", y="recent_points_H")
    plt.title("Recent home form vs match outcome")
    plt.tight_layout()
    plt.show()